# 🖥️ Unsloth Studio UI on Colab (노코드)

코랩에서 **Unsloth Studio 웹 UI**를 띄워 클릭만으로 모델을 받고 파인튜닝합니다.
**공식 Colab 방식** 그대로입니다 (unslothai/unsloth/studio).

> ⚠️ 먼저 **런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 **런타임 → 모두 실행**!

참고: `pip install unsloth-studio` 나 `curl install.sh` 는 코랩용이 **아닙니다**.
코랩에서는 아래처럼 레포를 clone하고 `setup.sh` → `colab.start()` 로 띄웁니다.


## 0. GPU 확인 (T4 인지 확인)


In [ ]:
!nvidia-smi


## 0.5. 환경 변수 고정 (중요!)
Colab의 matplotlib **inline 백엔드**가 Studio와 충돌해 시작 시
`ValueError: ... backend; 'module://matplotlib_inline...'` 에러가 납니다.
headless 백엔드(`agg`)로 덮어써서 방지합니다. **setup·start 보다 먼저 실행**하세요.


In [ ]:
import os
# Studio venv 의 matplotlib 가 Colab inline 백엔드를 못 읽어 죽는 문제 방지
os.environ["MPLBACKEND"] = "agg"
print("MPLBACKEND =", os.environ["MPLBACKEND"])


## 1. Setup — Unsloth 레포 clone & 설치
공식 setup 스크립트를 실행합니다. (수 분 소요, 의존성 설치)
끝에 'Start Unsloth Studio now?' 자동 실행이 한 번 죽어도(무시 가능) 설치는 완료됩니다 — 2번 셀에서 다시 시작합니다.


In [ ]:
!git clone --depth 1 --branch main https://github.com/unslothai/unsloth.git
%cd /content/unsloth
# 자동 실행(--local 의 prompt) 충돌을 피하려 MPLBACKEND 를 셸에도 명시
!chmod +x studio/setup.sh && MPLBACKEND=agg ./studio/setup.sh --local


## 2. Studio 시작
`start()` 가 서버 실행 + 포트 노출(Open 버튼)을 자동으로 처리합니다.
실행 후 나오는 **"Open Unsloth Studio"** 링크/박스를 클릭하세요.


In [ ]:
import os, sys
os.environ["MPLBACKEND"] = "agg"   # 안전하게 한 번 더 고정
BACKEND = "/content/unsloth/studio/backend"
# 자가 점검: setup(1번 셀)을 안 돌렸으면 colab 모듈을 못 찾음
if not os.path.isdir(BACKEND):
    raise RuntimeError("❌ 먼저 1번 'Setup' 셀을 실행하세요. (/content/unsloth 가 없습니다)")
if BACKEND not in sys.path:
    sys.path.insert(0, BACKEND)   # 'No module named colab' 방지
from colab import start
start()


## 3. UI에서 모델 다운로드 & 학습

열린 Studio 화면에서:
1. **모델 선택칸**에 HuggingFace 모델명 입력 → 자동 다운로드
   - 추천(초경량): `unsloth/Qwen2.5-1.5B-Instruct`, `unsloth/Qwen2.5-0.5B-Instruct`, `unsloth/Llama-3.2-1B-Instruct`
2. **데이터** 업로드(CSV/JSONL). 우리 20시나리오 데이터를 쓰려면 아래 4번 셀로 먼저 생성.
3. LoRA/하이퍼파라미터 조정 → **Train** → **Model Arena** 비교 → **Export**(GGUF 등).

각 설정의 의미·조절법은 레포의 `docs/LEARNING.md` 치트시트 참고.


## 4. (선택) 우리 레포 20시나리오 데이터 생성 → UI에 업로드
Studio UI의 데이터 업로드칸에 넣을 `hf_train.jsonl` 을 만듭니다.


In [ ]:
%cd /content
!git clone -q https://github.com/xide-projext/edu-llm-colab-unsloth.git
%cd /content/edu-llm-colab-unsloth
!pip install -q datasets
!python scripts/fetch_hf_datasets.py --per-source 2000 --val-ratio 0.05
print("생성됨: /content/edu-llm-colab-unsloth/data/hf_train.jsonl → 좌측 파일탭에서 다운로드 후 Studio UI에 업로드")


---
### 문제 해결
- **`ValueError: Key backend: 'module://matplotlib_inline...'`**: 0.5번 셀(`MPLBACKEND=agg`)을 먼저 실행했는지 확인. 이미 죽었다면 0.5번 → 2번 순서로 다시 실행.
- **Open 링크가 에러/빈 화면**: 쿠키·애드블록 때문. 2번 셀을 다시 실행하거나, 출력 박스 아래로 스크롤하면 UI가 직접 보입니다. (알려진 버그 unslothai/unsloth#4516)
- **setup.sh 실패**: 1번 셀 로그 확인 후 재실행. 런타임이 **T4 GPU** 인지 다시 확인.
- **OOM**: UI에서 더 작은 모델(0.5B), batch/seq 축소 (LEARNING.md 치트시트).
- **코드로 재현 학습**: UI 대신 `unsloth_edu_finetune.ipynb` 사용 (권장 — 재현·버전관리).
